# Hyperparameter Tuning for Autoencoders

## Overview
This notebook evaluates different architectures and compression ratios for two types of autoencoders:

1. **Concrete Autoencoder (CAE)** - Uses a concrete/Gumbel-softmax distribution for feature selection during training
2. **Feature Selection Autoencoder (FSAE)** - Implements feature selection through learnable gates

## Objectives
- Compare various autoencoder architectures with different hidden layer configurations
- Test different compression ratios to find optimal dimensionality reduction
- Evaluate reconstruction performance and feature selection quality

## Setup
Run all cells sequentially. Make sure you have a `.env` file with your configuration!

In [ ]:
import os
import shutil
import time

import pandas as pd
import boto3
import numpy as np
from dotenv import load_dotenv
import pyarrow.dataset as ds

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split

from boruta import BorutaPy

load_dotenv()

In [ ]:
def get_results(hist, autoencoder):
  # Validation loss
  val_loss = hist.history["val_loss"][-1]

  # Test loss
  X_reconstructed = autoencoder.predict(X_test.values)
  test_loss = float(np.mean(np.square(X_test.values - X_reconstructed)))

  # Per feature
  errors = np.square(X_test - X_reconstructed)
  feature_mse = np.mean(errors, axis=0)
  features_95_percentile = np.percentile(feature_mse, 95)

  # Per pixel
  errors = np.square(X_test - X_reconstructed)
  pixel_mse = np.mean(errors, axis=1)
  pixel_95_percentile = np.percentile(pixel_mse, 95)

  # Per Critical Wavelengths
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelength_mse = feature_mse[critical_wavelengths]
  critical_99_percentile = np.percentile(critical_wavelength_mse, 99)


  performance = {
      "val_loss": val_loss,
      "test_loss": test_loss,
      "features_95_percentile": features_95_percentile,
      "pixel_95_percentile": pixel_95_percentile,
      "critical_wavelength_99_percentile": critical_99_percentile
  }

  return pd.Series(performance)

# 1. Import Data

In [ ]:
bucket = os.getenv("BUCKET_NAME")
results_bucket = os.getenv("HYPERPARAMETERS_RESULTS_BUCKET_NAME")

column_path = f"s3://{bucket}/headers.parquet"
columns =  pd.read_parquet(column_path)['0'].values

s3_folder = f"s3://{bucket}/samples/"
dataset = ds.dataset(s3_folder, format="parquet")
toscore = dataset.to_table().to_pandas()

toscore.columns = columns

bad_wavelengths = [c for c in toscore.columns if c < 320 or (c > 590 and c < 610)]

bad_wavelengths

X = toscore.drop(columns=bad_wavelengths).dropna()
X

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# 2. CAE Hyperparameter Tuning

In [ ]:
class ConcreteSelect(layers.Layer):
    def __init__(self, k, input_dim, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self.input_dim = input_dim
        self.temperature = temperature

    def build(self, input_shape):
        self.logits = self.add_weight(
            shape=(self.k, self.input_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='logits'
        )

    def call(self, inputs, training=None):
        if training:
            uniform = tf.random.uniform(tf.shape(self.logits), minval=0, maxval=1)
            gumbel = -tf.math.log(-tf.math.log(uniform + 1e-20) + 1e-20)
            noisy_logits = (self.logits + gumbel) / self.temperature
            scores = tf.nn.softmax(noisy_logits, axis=-1)
        else:
            scores = tf.one_hot(tf.argmax(self.logits, axis=-1), depth=self.input_dim)
        return tf.matmul(inputs, tf.transpose(scores))

    def get_config(self):
        config = super().get_config()
        config.update({
            "k": self.k,
            "input_dim": self.input_dim,
            "temperature": self.temperature
        })
        return config


class ColumnSelector(layers.Layer):
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = tf.constant(indices, dtype=tf.int32)

    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"indices": self.indices.numpy().tolist()})
        return cfg


In [ ]:
def get_results(hist, autoencoder, layers, is_symmetric):
  # Validation loss
  val_loss = hist.history["val_loss"][-1]

  # Test loss
  X_reconstructed = autoencoder.predict(X_test.values)
  test_loss = float(np.mean(np.square(X_test.values - X_reconstructed)))

  # Per feature
  errors = np.square(X_test - X_reconstructed)
  feature_mse = np.mean(errors, axis=0)
  features_95_percentile = np.percentile(feature_mse, 95)

  # Per pixel
  errors = np.square(X_test - X_reconstructed)
  pixel_mse = np.mean(errors, axis=1)
  pixel_95_percentile = np.percentile(pixel_mse, 95)

  # Per Critical Wavelengths
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelength_mse = feature_mse[critical_wavelengths]
  critical_99_percentile = np.percentile(critical_wavelength_mse, 99)


  performance = {
      "layers": layers,
      "symmetric_architecture": is_symmetric,
      "val_loss": val_loss,
      "test_loss": test_loss,
      "features_95_percentile": features_95_percentile,
      "pixel_95_percentile": pixel_95_percentile,
      "critical_wavelength_99_percentile": critical_99_percentile
  }

  return pd.Series(performance)

## 2.1 Asymmetric Architecture

### 2.1.1 16 Neurons

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

k = 16

input_dim = X_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(32, activation="relu")(encoded)
decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(cae_hist, cae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 2.1.2 Asymmetric_24

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

k = 24

input_dim = X_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(32, activation="relu")(encoded)
decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(cae_hist, cae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 2.1.3 Asymmetric_32 (Base model)

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

k = 32

input_dim = X_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(cae_hist, cae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",v
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 2.1.4 Asymetric 48

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

k = 48

input_dim = X_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(256, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(cae_hist, cae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 2.1.5 Asymmetric_64

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

k = 64

input_dim = X_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(256, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=60,
    batch_size=16,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

version = "v2"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(cae_hist, cae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

## 2.2 Symmetric

In [ ]:
for k in [16, 24, 32, 48, 64]:
    num_features = 277
    X_scaled = X_train.dropna().values

    input_dim = X_scaled.shape[1]

    input_layer = keras.Input(shape=(input_dim,))
    encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)
    decoded = layers.Dense(input_dim, activation="sigmoid")(encoded)

    cae = keras.Model(inputs=input_layer, outputs=decoded)
    cae.compile(optimizer="adam", loss="mse")

    cae_hist = cae.fit(
        X_scaled, X_scaled,
        epochs=60,
        batch_size=32,
        shuffle=True,
        validation_split=0.2
    )

    encoder = keras.Model(inputs=input_layer, outputs=encoded)
    X_encoded = encoder.predict(X_scaled)

    # Reconstruct
    X_reconstructed = cae.predict(X_scaled)

    version = "v1"
    architecture = "symmetric_architecture"
    encoder_size = k

    performance = get_results(cae_hist, cae, encoder_size, 1).to_frame()

    validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{architecture}/{encoder_size}/{version}/score.parquet"
    weights_path = f"concrete-auto-encoder/{architecture}/{str(encoder_size)}/{version}/cae_savedmodel.zip"

    performance.to_parquet(
        validation_path,
        index=True,
        engine="pyarrow",
    )

    os.makedirs("cae_artifacts", exist_ok=True)
    cae.save_weights("cae_artifacts/cae_weights.weights.h5")

    zip_path = shutil.make_archive("cae_savedmodel", "zip", "cae_artifacts")
    s3 = boto3.client("s3")
    s3.upload_file(zip_path, results_bucket, weights_path)

## 3. FSAE

### 3.1 32 Encoded Neurons

In [ ]:
class ColumnSelector(layers.Layer):
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = tf.constant(indices, dtype=tf.int32)

    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"indices": self.indices.numpy().tolist()})
        return cfg


y_train = None
y = None

k = 32
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

### 3.1.1 Asymmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]
k = 32

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(64, activation="relu")(encoded)
x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=60,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v2"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 3.1.2 Symmetric_ 32

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]
k = 32

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)
out = layers.Dense(input_dim, activation="linear")(encoded)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "symmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

## 3.2. 16 Neurons

In [ ]:
y_train = None
y = None

k = 16
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

### 3.2.1 Asymmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(32, activation="relu")(encoded)
x = layers.Dense(64, activation="relu")(encoded)
x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 3.2.2 Symmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)
out = layers.Dense(input_dim, activation="linear")(encoded)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "symmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

## 3.3 24 Neurons

In [ ]:
y_train = None
y = None

k = 24
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

### 3.3.1 Asymmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(32, activation="relu")(encoded)
x = layers.Dense(64, activation="relu")(encoded)
x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 3.3.2 Symmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

out = layers.Dense(input_dim, activation="linear")(encoded)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "symmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

## 3.4 48 Neurons

In [ ]:
y_train = None
y = None

k = 48
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

### 3.4.1 Asymmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(64, activation="relu")(encoded)
x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 3.4.2 Symmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

out = layers.Dense(input_dim, activation="linear")(encoded)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "symmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

## 3.5 64 Neurons

In [ ]:
y_train = None
y = None

k = 64
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

### 3.5.1 Asymmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(64, activation="relu")(encoded)
x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=60,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "asymmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

### 3.5.2 Symmetric

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

out = layers.Dense(input_dim, activation="linear")(encoded)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=60,
    batch_size=32,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

version = "v1"
architecture = "symmetric_architecture"
encoder_size = k

performance = get_results(fsae_hist, fsae, encoder_size, 0).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{architecture}/{encoder_size}/{version}/score.parquet"
weights_path = f"fsae/{architecture}/{str(encoder_size)}/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "fsae_artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance